# Backend

> How a step's transmissions are simulated.

In [ ]:
#| default_exp backend

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

from __future__ import annotations

import math
from abc import ABC, abstractmethod
from typing import List, Sequence, Tuple

import numpy as np

from comm_core.block import Block, Pipeline
from comm_core.core import Transmission, TransmissionResult
from comm_core.fading import effective_coefficient
from comm_core.topology import Link
from comm_core.channel import Channel


class CommunicationBackend(ABC):
    "Carries out the transmissions of each step: the interface the `Network` simulates them through."

    @abstractmethod
    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:
        pass

    def transmit_all(
        self,
        transmissions: List[Tuple[Transmission, Link]], # The transmissions of one step, with their links
    ) -> List[TransmissionResult]:
        "The transmissions of one step, all at once (they may affect each other, e.g. interfere); by default one at a time."
        return [self.transmit(transmission, link) for transmission, link in transmissions]

    def reset(self, seed: int | None = None) -> None:
        "Start over; `seed` reseeds the randomness of the simulation."
        pass

    def step(self, time: float) -> None:
        pass


class AnalyticalBackend(CommunicationBackend):
    "A backend that hands each transmission to a `Channel` model."

    def __init__(
        self,
        channel: Channel,
    ):
        self.channel = channel

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:

        return self.channel.transmit(
            transmission,
            link,
        )

    def transmit_all(self, transmissions):
        "The step's transmissions; in one call when the channel can (`transmit_batch`)."
        if hasattr(self.channel, 'transmit_batch'):
            return self.channel.transmit_batch(transmissions)
        return [self.channel.transmit(transmission, link) for transmission, link in transmissions]

    def reset(self, seed: int | None = None) -> None:
        self.channel.reset(seed)

    def step(self, time: float) -> None:
        self.channel.update(time)

## Pipelines of blocks

`PipelineBackend` runs a step's transmissions through blocks (see `block`). It calls them with the step's inputs
(`transmission`, `link`, `sender`, `receiver`, `size_bits`, `group`) and the constants it is given (e.g.
`noise_power`), and makes a `TransmissionResult` of each row of the outputs:

* **Required:** `success`. Missing optional outputs take defaults: the payload as sent if it arrived, no latency
  or energy, `size_bits` bits.
* **Radio:** with `h` and `power`, the results get the SNR (the SINR with `interference`) and the metadata
  `channel_gain_db` and `tx_power`. Rows not `active` are outages (`outage`, nothing sent); rows with
  `half_duplex` were sent, but their receiver was transmitting.
* **Metadata:** `info` (a dict per row) is added to the metadata.

The radio channels (see `radio`) are pipelines of this kind, so a pipeline of your own can reuse their blocks:
replace one, add one, or wrap a Sionna block in `Named`.

In [ ]:
#| export
STEP_INPUTS = ('transmission', 'link', 'sender', 'receiver', 'size_bits', 'group')


def step_inputs(
    transmissions: List[Tuple[Transmission, Link]], # The transmissions of a step, with their links
) -> dict:
    "The named inputs of a step: `STEP_INPUTS`, one row per transmission (see the names in `block`)."
    return {
        'transmission': [t for t, _ in transmissions],
        'link': [l for _, l in transmissions],
        'sender': [t.sender for t, _ in transmissions],
        'receiver': [t.receiver for t, _ in transmissions],
        'size_bits': np.array([t.message.size_bits or 0 for t, _ in transmissions], dtype=np.int64),
        'group': [t.message.metadata.get('broadcast') for t, _ in transmissions],
    }


def _db(x: float) -> float:
    return 10 * math.log10(x) if x > 0 else -math.inf


def to_results(
    state: dict, # The named arrays of a step after its blocks, with its inputs
) -> List[TransmissionResult]:
    "The `TransmissionResult` of each transmission, from the named outputs (see above)."
    transmissions = state['transmission']
    active, half_duplex = state.get('active'), state.get('half_duplex')
    radio = 'h' in state and 'power' in state
    noise_power = state.get('noise_power', 1.0)
    error = state['error_rate'] if 'error_rate' in state else state.get('per')
    results = []
    for k, t in enumerate(transmissions):
        deaf = half_duplex is not None and bool(half_duplex[k])
        sent = active is None or bool(active[k]) or deaf
        metadata, snr_db = {}, None
        if radio:
            gain, power = abs(effective_coefficient(state['h'][k])) ** 2, float(state['power'][k])
            snr_db = math.inf if gain == math.inf else _db(power * gain / noise_power)
            metadata = {'channel_gain_db': _db(gain), 'tx_power': power if sent else 0.0}
        if not sent:                                    # an outage: nothing on the air
            results.append(TransmissionResult(success=False, transmission=t, payload=None, size_bits=0, snr_db=snr_db,
                                              metadata={**metadata, 'outage': True}))
            continue
        num_bits = int((state['num_bits'] if 'num_bits' in state else state['size_bits'])[k])
        latency = float(state['latency'][k]) if 'latency' in state else 0.0
        energy = float(state['energy'][k]) if 'energy' in state else 0.0
        if deaf:                                        # sent and paid for, but the receiver was transmitting
            results.append(TransmissionResult(success=False, transmission=t, payload=None, latency=latency, size_bits=num_bits,
                                              energy=energy, snr_db=snr_db, error_rate=1.0, metadata={**metadata, 'half_duplex': True}))
            continue
        if 'success' not in state:
            raise ValueError("no block gives `success`")
        interference = float(state['interference'][k]) if 'interference' in state else 0.0
        if radio and interference and snr_db != math.inf:
            snr_db = _db(power * gain / (noise_power + interference))
        success = bool(state['success'][k])
        payload = state['payload'][k] if 'payload' in state else (t.message.payload if success else None)
        results.append(TransmissionResult(
            success=success,
            transmission=t,
            payload=payload,
            latency=latency,
            size_bits=num_bits,
            energy=energy,
            snr_db=snr_db,
            error_rate=None if error is None else float(error[k]),
            metadata={**metadata, **({'interference_power': interference} if interference else {}),
                      **(state['info'][k] if 'info' in state else {})},
        ))
    return results


class PipelineBackend(CommunicationBackend):
    "A step's transmissions through blocks: the step's named inputs in, a `TransmissionResult` per row out."

    def __init__(
        self,
        blocks: Pipeline | Sequence[Block], # The blocks (checked against the step's inputs and the constants)
        **constants, # More named inputs, the same at every step, e.g. noise_power=1e-9
    ):
        self.constants = constants
        self.pipeline = blocks if isinstance(blocks, Pipeline) else Pipeline(blocks, inputs=STEP_INPUTS + tuple(constants))

    def transmit(self, transmission: Transmission, link: Link) -> TransmissionResult:
        "One transmission alone."
        return self.transmit_all([(transmission, link)])[0]

    def transmit_all(
        self,
        transmissions: List[Tuple[Transmission, Link]], # The transmissions of one step, with their links
    ) -> List[TransmissionResult]:
        "The transmissions of one step, through the blocks at once."
        if not transmissions:
            return []
        return to_results(self.pipeline(**step_inputs(transmissions), **self.constants))

    def reset(self, seed: int | None = None) -> None:
        "A new episode: the blocks are reseeded (see `Block.reset`)."
        self.pipeline.reset(seed)

    def step(self, time: float) -> None:
        "A new round: passed to the blocks (`Block.update`)."
        self.pipeline.update(time)

    def set_positions(self, positions) -> None:
        "Where the nodes are, for the blocks that need it (e.g. `RayTracedFading`)."
        for block in self.pipeline.walk():
            if block is not self.pipeline and hasattr(block, 'set_positions'):
                block.set_positions(positions)

In [ ]:
show_doc(PipelineBackend)

In [ ]:
show_doc(to_results)

In [ ]:
from comm_core.core import Message, Node
from comm_core.network import Network
from comm_core.topology import FullMeshTopology
from comm_core.protocol import IdealProtocol

In [ ]:
# a channel of two blocks: packets up to a size arrive, after their time on the air
class Small(Block):
    outputs = ('success',)
    def __init__(self, limit):
        self.limit = limit
    def call(self, size_bits):
        return size_bits <= self.limit

class Rate(Block):
    outputs = ('latency',)
    def call(self, size_bits, rate_bps):
        return size_bits / rate_bps

backend = PipelineBackend([Small(1000), Rate()], rate_bps=1e6)
net = Network([Node('a'), Node('b')], FullMeshTopology(['a', 'b']), IdealProtocol(), backend)
net.send(Message('a', 'b', 'x', size_bits=800)); net.send(Message('a', 'b', 'y', size_bits=5000))
r1, r2 = net.step()
assert r1.success and r1.payload == 'x' and r1.latency == 8e-4 and r1.size_bits == 800 and r1.snr_db is None
assert not r2.success and r2.payload is None and r2.metadata == {}
assert net.receive('b')[0].payload == 'x' and backend.transmit_all([]) == []
try:
    PipelineBackend([Rate()]); raise AssertionError('no error')        # rate_bps: neither an input nor a constant
except ValueError as e:
    assert 'rate_bps' in str(e)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()